## Setting up AIP Keys and Connecting to OpenAI and Local LLM:

In [8]:
# Imports:

import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown

In [6]:
load_dotenv(override= True)

api_key = os.getenv('OPENAI_API_KEY')
ollama_url = 'http://172.23.37.63:11434/v1'

openai = OpenAI()
ollama = OpenAI(base_url= ollama_url,api_key= 'fake_key')

In [4]:
# Checking OpenAI API Connection:

response = openai.chat.completions.create(model= 'gpt-5-mini',
                                          messages= [{
                                              'role': 'user',
                                              'content': 'Hi, My name is Shailya. May I know your name?'
                                          }])

print(response.choices[0].message.content)

Hi Shailya — nice to meet you! I'm ChatGPT, an AI assistant. How can I help you today?


In [7]:
# Checking OLLAMA API Connection:

response = ollama.chat.completions.create(model= 'llama3.1:8b',
                                          messages= [{
                                              'role': 'user',
                                              'content': 'Hi, My name is Shailya. May I know your name?'
                                          }])

print(response.choices[0].message.content)

Hello Shailya! I'm happy to meet you. I don't have a personal name, as I'm an artificial intelligence designed to assist and communicate with users. I'm often referred to as a chatbot or a conversational AI. I'll be happy to chat with you as "Assistant" if you prefer a name to address me by! What brings you here today?


## Training vs Inference Time Scaling:

For the first few years of the Generative AI boom, the industry relied entirely on one method to make models smarter: **Training-Time Scaling**. In late 2024, the ceiling of that method started to show, leading to the breakthrough of **Inference-Time Scaling** (Test-Time Compute).

Understanding the economic and architectural differences between these two is critical for deploying the right model for the right job.

## 1. Training-Time Scaling (The "Brute Force" Era)
**What it is:** Making a model smarter by throwing exponentially more data and compute at it *before* it is ever released to the public.

**How it works:** The industry follows "Scaling Laws" (specifically the Chinchilla Scaling Laws). The rule is simple: if you want a model to be twice as smart, you need to increase its parameter count and the size of its training dataset, which requires massive clusters of GPUs running for months.

* **The Economics:** All the cost is upfront. Training a frontier model costs hundreds of millions of dollars.
* **The User Experience:** Fast and cheap. Once the model is trained, generating an answer (inference) takes milliseconds because the model has already "memorized" the patterns.
* **The Limitation:** We are physically running out of high-quality human text on the internet to train these massive models on, and building larger data centers is becoming constrained by power grids.

## 2. Inference-Time Scaling (The "Thinking" Era)
**What it is:** Making a model smarter by giving it more compute power *after* the user asks a question, right in the moment it is generating the answer.

**How it works:**
Instead of relying purely on what it memorized during training, the model uses **Test-Time Compute**. When given a complex logic puzzle or a deep coding architecture problem, it initiates a hidden "Chain of Thought."
It uses computational power to generate thousands of invisible tokens—testing hypotheses, catching its own math errors, and correcting its logic—before printing the final output.

* **The Economics:** The cost is shifted to the user query. The upfront training might be cheaper, but running the API call is incredibly computationally expensive because the model might generate 10,000 hidden tokens just to give you a 100-word answer.
* **The User Experience:** Slow and deliberate. You might wait 10 to 30 seconds for a response.
* **The Advantage:** This allows much smaller models to punch massively above their weight class on complex problems by simply "thinking" longer.

## Summary Comparison

| Feature | Training-Time Scaling | Inference-Time Scaling |
| :--- | :--- | :--- |
| **When does the heavy lifting happen?** | Months before the user ever sees it. | The exact moment the user hits "Send." |
| **How does it get smarter?** | Reading more data, adding more parameters. | "Thinking" longer, generating hidden CoT tokens. |
| **Best used for:** | General knowledge, creative writing, standard code generation. | Advanced mathematics, Theory of Mind logic, complex data pipelines. |
| **Examples:** | GPT-4o, Llama 3.1 8B, Claude 3.5 Sonnet | OpenAI o1/o3, DeepSeek R1 |

In [12]:
# Training Vs Inference Time Scaling - Example:

# Easy Puzzle - Low Reasoning:
easy_puzzle = [
    {"role": "user", "content":
        "You toss 2 coins. One of them is heads. What's the probability the other is tails? Answer with the probability only."},
]

response = openai.chat.completions.create(model= 'gpt-5-nano', messages= easy_puzzle, reasoning_effort= 'minimal')
display(Markdown(response.choices[0].message.content))

1/2

In [13]:
# Easy Puzzle - High Reasoning:
easy_puzzle = [
    {"role": "user", "content":
        "You toss 2 coins. One of them is heads. What's the probability the other is tails? Answer with the probability only."},
]

response = openai.chat.completions.create(model= 'gpt-5-nano', messages= easy_puzzle, reasoning_effort= 'high')
display(Markdown(response.choices[0].message.content))

2/3

In [14]:
# Hard Puzzle - Minimal Reasoning:
hard = """
On a bookshelf, two volumes of Pushkin stand side by side: the first and the second.
The pages of each volume together have a thickness of 2 cm, and each cover is 2 mm thick.
A worm gnawed (perpendicular to the pages) from the first page of the first volume to the last page of the second volume.
What distance did it gnaw through?
"""
hard_puzzle = [
    {"role": "user", "content": hard}
]

response = openai.chat.completions.create(model= 'gpt-5-nano', messages= hard_puzzle, reasoning_effort= 'minimal')
display(Markdown(response.choices[0].message.content))

Let’s map the books and their dimensions.

- Each volume has pages total thickness: 2 cm = 20 mm.
- Each cover thickness: 2 mm.
- There are two volumes side by side: Volume 1 (V1) followed by Volume 2 (V2).
- The worm starts at the first page of V1 and ends at the last page of V2, moving perpendicular to the pages (i.e., horizontally along the shelf).

Consider the order from left to right:
- Cover of V1 (left cover) 2 mm
- Pages of V1: 20 mm
- Cover of V1 (right cover) 2 mm
- Cover of V2 (left cover) 2 mm
- Pages of V2: 20 mm
- Right cover of V2: 2 mm

The worm starts at the first page of V1, which is just after the left cover of V1, i.e., at the inner surface of V1’s left page start boundary. It ends at the last page of V2, which is just before the right cover of V2.

Thus it travels through:
- The remainder of V1’s pages plus possibly the right cover of V1, and the entire left cover of V2 and entire pages of V2 up to the last page (but not through V2’s right cover).

But a standard interpretation of this classic problem is to count the physical material between the first page of V1 and the last page of V2, along the shelf, including:
- The rest of V1’s pages after the first page: essentially the full 20 mm of V1 pages.
- The right cover of V1: 2 mm.
- The left cover of V2: 2 mm.
- The entire pages of V2: 20 mm.

Add them: 20 + 2 + 2 + 20 = 44 mm.

Therefore, the worm gnawed through 4.4 cm.

In [15]:
# Hard Puzzle - Low Reasoning:
hard = """
On a bookshelf, two volumes of Pushkin stand side by side: the first and the second.
The pages of each volume together have a thickness of 2 cm, and each cover is 2 mm thick.
A worm gnawed (perpendicular to the pages) from the first page of the first volume to the last page of the second volume.
What distance did it gnaw through?
"""
hard_puzzle = [
    {"role": "user", "content": hard}
]

response = openai.chat.completions.create(model= 'gpt-5-nano', messages= hard_puzzle, reasoning_effort= 'low')
display(Markdown(response.choices[0].message.content))

Answer: 4.4 cm

Reasoning:
- Each volume has: pages = 20 mm thick; each cover = 2 mm.
- Start at the first page of the first volume (the very beginning of the pages).
- From there to the right outer face of the first volume: the worm goes through the rest of the first volume, i.e., 20 mm (remaining pages) + 2 mm (right cover) = 22 mm.
- To enter the second volume, it must pass through the left cover of the second volume: 2 mm.
- Then through the pages of the second volume up to the last page: 20 mm.

Total distance = 22 mm + 2 mm + 20 mm = 44 mm = 4.4 cm.

In [16]:
# Hard Puzzle - High Reasoning:
hard = """
On a bookshelf, two volumes of Pushkin stand side by side: the first and the second.
The pages of each volume together have a thickness of 2 cm, and each cover is 2 mm thick.
A worm gnawed (perpendicular to the pages) from the first page of the first volume to the last page of the second volume.
What distance did it gnaw through?
"""
hard_puzzle = [
    {"role": "user", "content": hard}
]

response = openai.chat.completions.create(model= 'gpt-5-nano', messages= hard_puzzle, reasoning_effort= 'high')
display(Markdown(response.choices[0].message.content))

4.4 cm (which is 44 mm).

Reason:
- Each volume has pages thickness = 2 cm.
- Each cover thickness = 2 mm = 0.2 cm.
- Two volumes touch at the seam between vol1’s back cover and vol2’s front cover.
- The worm’s path from the first page of vol1 to the last page of vol2 goes through: vol1 pages (2 cm), vol1 back cover (0.2 cm), vol2 front cover (0.2 cm), vol2 pages (2 cm).
- Total = 2 + 0.2 + 0.2 + 2 = 4.4 cm.

In [17]:
# Hard Puzzle - High Reasoning (Different Model):
hard = """
On a bookshelf, two volumes of Pushkin stand side by side: the first and the second.
The pages of each volume together have a thickness of 2 cm, and each cover is 2 mm thick.
A worm gnawed (perpendicular to the pages) from the first page of the first volume to the last page of the second volume.
What distance did it gnaw through?
"""
hard_puzzle = [
    {"role": "user", "content": hard}
]

response = openai.chat.completions.create(model= 'gpt-5-mini', messages= hard_puzzle, reasoning_effort= 'high')
display(Markdown(response.choices[0].message.content))

4 mm (0.4 cm).

Reason: on the shelf the first page of Vol. 1 lies just under its front cover and the last page of Vol. 2 just under its back cover. With the two volumes side by side the worm only has to chew through those two facing covers: 2 mm + 2 mm = 4 mm.